# Digital Twin rApp — Sionna RT + OAI Walkthrough

Same pipeline and **same scene** as `dtrapp_sionna_walkthrough.ipynb` (the Sionna SYS
version), so you can run both side by side and compare.

The only difference is **stage 5**: instead of the Sionna SYS link model, the
SINR → throughput mapping comes from **OpenAirInterface's real PHY** (measured by
`nr_dlsim`, stored in `oai/sinr_throughput_table.json`). Everything else — OSM scene,
network, Sionna RT channel, multi-cell SINR, scheduling — is identical.

## 0. Setup

In [ ]:
import os

# Force CPU: hide the GPU so Sionna RT / torch fall back to CPU.
# MUST be set before importing sionna. Comment out to use the GPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")

# Install dependencies if missing (uncomment to run once):
# %pip install sionna sionna-rt numpy pyyaml matplotlib pandas

%matplotlib inline
import sys
import glob
import math

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image

# ----------------------- EDIT THESE PATHS -----------------------
REPO_ROOT = os.path.abspath(os.environ.get("DTRAPP_REPO", ".."))
SCENE_DIR = os.path.join(REPO_ROOT, "output", "scene")
CONFIG_PATH = os.path.join(REPO_ROOT, "configs", "example.yaml")
# ----------------------------------------------------------------

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# Set True if you are NOT in a notebook GUI (renders images instead of the widget).
no_preview = False

print("REPO_ROOT :", REPO_ROOT)
print("SCENE_DIR :", SCENE_DIR)
print("CONFIG    :", CONFIG_PATH)
assert os.path.exists(os.path.join(SCENE_DIR, "scene.xml")), (
    "scene.xml not found - generate it first, e.g. "
    "`python3 -m dtrapp.runner.cli configs/example.yaml`."
)

## 1. Load the scenario config

In [ ]:
from dtrapp.config import SimulationConfig

config = SimulationConfig.from_yaml(CONFIG_PATH)
print("Loaded config:")
for k, v in config.to_dict().items():
    print(f"  {k}: {v}")

## 2. Stage 1 — load & visualize the 3D scene

In [ ]:
import sionna
from sionna.rt import load_scene, Camera

scene = load_scene(os.path.join(SCENE_DIR, "scene.xml"))
print("Scene loaded. Objects (merged by material):", len(scene.objects))


def read_ply(path):
    """Minimal ASCII-PLY reader for the meshes written by dtrapp."""
    with open(path, "r") as fh:
        lines = fh.read().splitlines()
    n_vert, header_end = 0, 0
    for i, ln in enumerate(lines):
        if ln.startswith("element vertex"):
            n_vert = int(ln.split()[-1])
        if ln.strip() == "end_header":
            header_end = i + 1
            break
    verts = [tuple(float(v) for v in ln.split()[:3])
             for ln in lines[header_end:header_end + n_vert]]
    return np.array(verts, dtype=float)


bldg_files = sorted(glob.glob(os.path.join(SCENE_DIR, "meshes", "bldg-*.ply")))
assert bldg_files, "No building meshes (bldg-*.ply) found in the scene folder."
all_xy = np.vstack([read_ply(p)[:, :2] for p in bldg_files])
extent_m = (
    float(all_xy[:, 0].min()), float(all_xy[:, 1].min()),
    float(all_xy[:, 0].max()), float(all_xy[:, 1].max()),
)
print(f"{len(bldg_files)} buildings")
print("extent_m (min_x, min_y, max_x, max_y) =",
      tuple(round(v, 2) for v in extent_m))

In [ ]:
# A handy aerial camera centered on the scene.
cx = 0.5 * (extent_m[0] + extent_m[2])
cy = 0.5 * (extent_m[1] + extent_m[3])
span = max(extent_m[2] - extent_m[0], extent_m[3] - extent_m[1])
aerial_cam = Camera(position=[cx, cy - span, span], look_at=[cx, cy, 0.0])

if no_preview:
    scene.render_to_file(camera=aerial_cam, filename="scene_overview.png",
                         resolution=[900, 600])
    display(Image("scene_overview.png"))
else:
    scene.preview()

## 3. Stage 2 — network data (cells + UEs)

In [ ]:
from dtrapp.network import RandomNetworkSource

network = RandomNetworkSource(config, extent_m).generate()
print(f"{len(network.cells)} cells, {len(network.ues)} UEs\n")

print("First 3 cells:")
for c in network.cells[:3]:
    pos = tuple(round(p, 1) for p in c.position)
    print(f"  {c.cell_id}: pos={pos} az={c.azimuth_deg:.1f}deg "
          f"P={c.tx_power_dbm}dBm f={c.carrier_freq_hz/1e9:.2f}GHz "
          f"BW={c.bandwidth_hz/1e6:.0f}MHz")

print("\nFirst 3 UEs:")
for u in network.ues[:3]:
    pos = tuple(round(p, 1) for p in u.position)
    print(f"  {u.ue_id}: pos={pos} demand={u.traffic_demand_mbps:.1f}Mbps "
          f"NF={u.noise_figure_db}dB")

In [ ]:
# Top-down map: building footprints + base stations (with sector azimuths) + UEs.
def plot_footprints(ax):
    for p in bldg_files:
        v = read_ply(p)
        ring = v[: len(v) // 2, :2]
        ring = np.vstack([ring, ring[0]])
        ax.fill(ring[:, 0], ring[:, 1], facecolor="0.85",
                edgecolor="0.5", lw=0.5, zorder=1)


fig, ax = plt.subplots(figsize=(9, 9))
plot_footprints(ax)
ax.scatter([u.position[0] for u in network.ues],
           [u.position[1] for u in network.ues],
           c="tab:blue", s=25, label="UE", zorder=3)
for c in network.cells:
    x, y, _ = c.position
    a = math.radians(c.azimuth_deg)
    ax.scatter([x], [y], c="red", marker="^", s=90, zorder=4)
    ax.arrow(x, y, 25 * math.cos(a), 25 * math.sin(a),
             head_width=6, color="red", zorder=4)
ax.scatter([], [], c="red", marker="^", s=90, label="cell (BS sector)")
ax.set_aspect("equal")
ax.set_xlabel("x (m, East)"); ax.set_ylabel("y (m, North)")
ax.set_title("Top-down: buildings, base stations (sectors), and UEs")
ax.legend(loc="upper right")
plt.show()

## 4. Stage 3 — Sionna RT propagation (path gain)

In [ ]:
from dtrapp.propagation import SionnaPropagationEngine

engine = SionnaPropagationEngine(os.path.join(SCENE_DIR, "scene.xml"), config)
path_gain_db = engine.compute_path_gain(network)   # (num_ues, num_cells), dB
print("path_gain_db shape:", path_gain_db.shape)
print("range: %.1f .. %.1f dB" % (path_gain_db.min(), path_gain_db.max()))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(path_gain_db, aspect="auto", cmap="viridis")
ax.set_xlabel("cell index"); ax.set_ylabel("UE index")
ax.set_xticks(range(len(network.cells)))
ax.set_xticklabels([c.cell_id for c in network.cells], rotation=90, fontsize=7)
ax.set_title("Per-link path gain (dB)")
fig.colorbar(im, label="path gain (dB)")
plt.show()

## 5. The OAI link curve (this is what replaces Sionna SYS)

This staircase was **measured by OpenAirInterface's real PHY** (`nr_dlsim`): for each
5G-NR MCS it records the lowest SNR at which the link is reliable. The engine maps
each UE's SINR to the highest MCS this curve allows. Regenerate it with
`python3 oai/characterize_link.py` (needs OAI built).

In [ ]:
from dtrapp.kpi import load_link_curve

curve = load_link_curve()
print("source:", curve.source)
print("bler_target:", curve.bler_target, " mcs_table:", curve.mcs_table_index)

xs = np.linspace(-10, 30, 400)
se = [curve.map_sinr(float(x))[1] for x in xs]
pts = curve.points
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(xs, se, color="tab:red", lw=2, label="SE used by engine")
ax.scatter([p["sinr_db"] for p in pts], [p["se_bps_per_hz"] for p in pts],
           c="k", s=20, zorder=3, label="OAI-measured points")
for p in pts[::3]:
    ax.annotate(f"MCS{p['mcs']}", (p["sinr_db"], p["se_bps_per_hz"]),
                textcoords="offset points", xytext=(2, 4), fontsize=7)
ax.set_xlabel("SINR (dB)"); ax.set_ylabel("spectral efficiency (bits/s/Hz)")
ax.set_title(f"OAI-measured SINR → MCS → SE curve\n{curve.source}")
ax.grid(alpha=0.3); ax.legend()
plt.show()

## 6. Stages 4–5 — SINR & throughput (OAI-mapped)

`compute_kpis` computes each UE's multi-cell SINR from the ray-traced channel (CFR),
maps it through the OAI curve above to get MCS + spectral efficiency, and shares each
cell's bandwidth among its UEs (proportional-fair = equal airtime on a static snapshot).

In [ ]:
from dtrapp.kpi import compute_kpis

cfr = engine.compute_cfr(network)
result = compute_kpis(network, cfr, config)

try:
    import pandas as pd
    ue_df = pd.DataFrame(result.ue_rows())
    cell_df = pd.DataFrame(result.cell_rows())
    print("Per-UE KPIs (first 10 rows):")
    display(ue_df.head(10))
    print("Per-cell KPIs:")
    display(cell_df)
    print("Mean UE throughput: %.2f Mbps" % ue_df["throughput_mbps"].mean())
except ImportError:
    ue_df = cell_df = None
    for u in result.ues[:10]:
        print(u)

In [ ]:
# Distributions of SINR and throughput across UEs.
sinr = np.array([u.sinr_db for u in result.ues])
tput = np.array([u.throughput_mbps for u in result.ues])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(sinr, bins=15, color="tab:orange", edgecolor="k")
axes[0].set_xlabel("SINR (dB)"); axes[0].set_ylabel("# UEs")
axes[0].set_title("SINR distribution")
axes[1].hist(tput, bins=15, color="tab:green", edgecolor="k")
axes[1].set_xlabel("throughput (Mbps)"); axes[1].set_ylabel("# UEs")
axes[1].set_title("UE throughput distribution (OAI-mapped)")
plt.tight_layout(); plt.show()

In [ ]:
# Map: each UE colored by throughput, labeled with id + Mbps, linked to serving cell.
cell_pos = {c.cell_id: c.position for c in network.cells}

fig, ax = plt.subplots(figsize=(11, 11))
plot_footprints(ax)
for u in result.ues:
    cp = cell_pos[u.serving_cell]
    ax.plot([u.x, cp[0]], [u.y, cp[1]], color="0.7", lw=0.5, zorder=2)
sc = ax.scatter([u.x for u in result.ues], [u.y for u in result.ues],
                c=tput, cmap="viridis", s=45, zorder=3)
for u in result.ues:
    ax.annotate(f"{u.ue_id} ({u.throughput_mbps:.1f})", (u.x, u.y),
                textcoords="offset points", xytext=(3, 3),
                fontsize=6, color="0.2", zorder=5)
for c in network.cells:
    x, y, _ = c.position
    a = math.radians(c.azimuth_deg)
    ax.scatter([x], [y], c="red", marker="^", s=90, zorder=4)
    ax.annotate(c.cell_id, (x + 30 * math.cos(a), y + 30 * math.sin(a)),
                fontsize=8, fontweight="bold", color="darkred",
                ha="center", va="center", zorder=6)
ax.set_aspect("equal")
ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)")
ax.set_title("UEs colored by throughput (label = ue_id and Mbps)")
fig.colorbar(sc, label="throughput (Mbps)")
plt.show()

In [ ]:
# Per-cell view: attached UEs and aggregate throughput.
ids = [c.cell_id for c in result.cells]
att = [c.num_attached for c in result.cells]
cmb = [c.throughput_mbps for c in result.cells]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(ids, att, color="tab:purple")
axes[0].set_title("UEs attached per cell"); axes[0].tick_params(axis="x", rotation=90)
axes[1].bar(ids, cmb, color="tab:green")
axes[1].set_title("Aggregate throughput per cell (Mbps)")
axes[1].tick_params(axis="x", rotation=90)
plt.tight_layout(); plt.show()

## 7. Stage 6 — write the output files

In [ ]:
from dtrapp.runner.output import write_outputs

written = write_outputs(result, config)
for p in written:
    print("wrote", p)

try:
    import pandas as pd
    display(pd.read_csv(os.path.join(config.output_dir, "ue_throughput.csv")).head())
except Exception as e:
    print("(install pandas to preview the CSV)", e)

## 8. (Optional) Run OAI over the *exact* ray-traced channel

The curve above characterizes OAI once. For maximum fidelity you can push the exact
ray-traced channel into a live OAI gNB↔UE link (single UE). After building OAI
(`bash oai/setup_oai.sh`), from the repo root:

```bash
# 1) generate the scene + channel export
python3 -m dtrapp.runner.cli configs/example.yaml

# 2) CFR -> exact taps + channel-enabled gNB conf
python3 oai/cfr_to_oai_channel.py output/channel \
    --base-conf ~/openairinterface5g/ci-scripts/conf_files/gnb.band78.106prb.rfsim.phytest-dora.conf

# 3) run OAI over the ray-traced channel and collect KPIs
OAI_RT_TAPS=output/channel/oai_rt_taps.txt \
CONF=output/channel/gnb_rtchan.conf bash oai/run_phytest.sh 30
python3 oai/collect_kpis.py oai_run/gnb.log oai_run/kpis.csv
```

The gNB log shows `[RT] injected ... taps` — OAI is transmitting over our channel.
See `oai/README.md` for the calibration caveat (UL power control masks path loss).

## Notes — side-by-side with the SYS notebook

- **Identical up to stage 4**: run this and `dtrapp_sionna_walkthrough.ipynb` on the
  same `output/scene` + same config (same seed) and stages 1–4 match exactly.
- **Stage 5 differs**: SYS uses `InnerLoopLinkAdaptation` + `PHYAbstraction` (a model);
  this uses the **OAI-measured** SINR→MCS curve. Compare the two `ue_throughput.csv`
  files: the `sinr_db` should be close (same physics), while `mcs`/`throughput_mbps`
  reflect OAI's real link adaptation vs. the SYS abstraction.
- The `mcs` column is new here (the MCS OAI would pick at that SINR).